# Profiler Exports for Batch Workflows
---

In the previous notebook we used TensorBoard to visualize PyTorch Profiler output interactively.  That works well in a JupyterLab environment with port forwarding, but most production training happens on **headless Slurm nodes** where setting up live TensorBoard access is friction-heavy.

Fortunately, PyTorch Profiler can export profiling data to several portable formats that you can download via `scp` and analyze locally.  This notebook covers four export options:

1. **Text tables** — human-readable summaries for logs and terminal output
2. **Chrome/Perfetto traces** — interactive timelines viewable in a local browser
3. **Memory timelines** — self-contained HTML for diagnosing OOM issues
4. **Flame graph stacks** — hierarchical views of where time is spent

By the end of this notebook you will know how to generate all four and when to use each.

## When to Use File-Based Exports

| Scenario | Best Export |
|----------|-------------|
| Quick check in Slurm logs | Text table |
| Deep-dive timeline analysis | Chrome trace → Perfetto |
| Debugging OOM or memory fragmentation | Memory timeline HTML |
| Understanding call-stack hotspots | Flame graph stacks |

All of these can be generated from the same profiler run — you just call different export methods after the profiling context closes.

## Setup: Create an Output Directory

We will write all export files to a single directory.  In a Slurm job you would typically write to `$SLURM_SUBMIT_DIR` or a scratch directory.

In [ ]:
from pathlib import Path

export_dir = Path("/workspace/reports/profiler_exports")
export_dir.mkdir(parents=True, exist_ok=True)
print(f"Export directory: {export_dir}")

## 1. Human-Readable Text Tables

The simplest export is a formatted text table showing the most expensive operations.  This is perfect for:

- Quick sanity checks in Slurm job logs
- Comparing runs without leaving the terminal
- Embedding in reports or documentation

After profiling completes, call `prof.key_averages().table()` to get a formatted string:

In [ ]:
import torch
import torchvision
import torchvision.transforms as T
import warnings

# Suppress profiler cycle warning
warnings.filterwarnings("ignore", message=".*Profiler clears events.*")

# Quick setup: model, data, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = torch.nn.CrossEntropyLoss()

transform = T.Compose([T.Resize(224), T.ToTensor(), T.Normalize((0.5,), (0.5,))])
dataset = torchvision.datasets.CIFAR10(root="../data", train=True, download=True, transform=transform)
loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
loader_iter = iter(loader)

In [ ]:
# Run profiler for a few steps
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    schedule=torch.profiler.schedule(wait=1, warmup=2, active=5, repeat=1),
    record_shapes=True,
) as prof:
    for step in range(10):
        try:
            x, y = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            x, y = next(loader_iter)
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        
        prof.step()

# Export text table sorted by CUDA time
table_text = prof.key_averages().table(sort_by="cuda_time_total", row_limit=20)
print(table_text)

In [ ]:
# Save to file for Slurm logs
with open(export_dir / "profile_summary.txt", "w") as f:
    f.write("=" * 80 + "\n")
    f.write("PyTorch Profiler Summary (sorted by cuda_time_total)\n")
    f.write("=" * 80 + "\n\n")
    f.write(table_text)

print(f"Saved: {export_dir / 'profile_summary.txt'}")

**Sorting options:** You can sort by `cpu_time_total`, `cuda_time_total`, `cpu_memory_usage`, `cuda_memory_usage`, `self_cpu_time_total`, or `self_cuda_time_total`.

---
## 2. Chrome / Perfetto Traces

For detailed timeline analysis, export a Chrome trace file.  This is the same data TensorBoard shows, but in a portable JSON format.

**How to view it:**
1. Download the `.json` or `.json.gz` file to your local machine
2. Open [ui.perfetto.dev](https://ui.perfetto.dev/) in your browser (recommended) or `chrome://tracing`
3. Drag and drop the file

Perfetto is a modern, WebGL-accelerated trace viewer that handles large PyTorch traces much better than the built-in Chrome tracer.

In [ ]:
# Export Chrome trace (gzip-compressed to save space)
trace_path = export_dir / "timeline_trace.json.gz"
prof.export_chrome_trace(str(trace_path))
print(f"Saved: {trace_path}")
print(f"View at: https://ui.perfetto.dev/")

**What you will see in Perfetto:**

- A timeline with CPU threads on the left and GPU streams below
- Each CUDA kernel as a colored bar showing its duration
- CPU-GPU synchronization points
- Zoom, pan, and click on any operation for details

This is the gold standard for deep-dive performance debugging when TensorBoard is not available.

---
## 3. Interactive Memory Timelines

If you are debugging Out-of-Memory (OOM) errors or trying to understand memory fragmentation, PyTorch can export a self-contained HTML file showing memory allocation over time.

**Requirements:**
- You must set `profile_memory=True` in the profiler configuration
- The export creates a standalone `.html` file you can open in any browser

In [ ]:
# Re-run with memory profiling enabled
loader_iter = iter(loader)

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    schedule=torch.profiler.schedule(wait=1, warmup=2, active=5, repeat=1),
    record_shapes=True,
    profile_memory=True,  # Required for memory timeline
) as prof_mem:
    for step in range(10):
        try:
            x, y = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            x, y = next(loader_iter)
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        
        prof_mem.step()

In [ ]:
# Export memory timeline HTML
memory_path = export_dir / "memory_timeline.html"
prof_mem.export_memory_timeline(str(memory_path), device="cuda:0")
print(f"Saved: {memory_path}")
print("Download and open in any browser to see interactive memory graph")

**What you will see:**

- Total GPU memory allocation over time
- Memory fragmentation visualization
- Which tensors are allocated at each point
- Interactive zoom and hover for details

This is invaluable when debugging OOM errors — you can see exactly when memory spikes and what tensors are responsible.

---
## 4. Flame Graph Stacks

Flame graphs show a hierarchical view of where execution time is spent across your model's call stack.  PyTorch exports this as a "folded stacks" text file that you can convert to an interactive SVG.

**Requirements:**
- You must set `with_stack=True` in the profiler configuration
- Use [FlameGraph](https://github.com/brendangregg/FlameGraph) tools to convert to SVG

In [ ]:
# Re-run with stack tracing enabled
loader_iter = iter(loader)

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    schedule=torch.profiler.schedule(wait=1, warmup=2, active=5, repeat=1),
    record_shapes=True,
    with_stack=True,  # Required for flame graphs
) as prof_stack:
    for step in range(10):
        try:
            x, y = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            x, y = next(loader_iter)
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        
        prof_stack.step()

In [ ]:
# Export folded stacks for flame graph generation
stacks_path = export_dir / "profiler_stacks.txt"
prof_stack.export_stacks(str(stacks_path), metric="self_cuda_time_total")
print(f"Saved: {stacks_path}")

**To generate a flame graph SVG:**

```bash
# Clone FlameGraph tools (one time)
git clone https://github.com/brendangregg/FlameGraph.git

# Generate SVG from the stacks file
./FlameGraph/flamegraph.pl profiler_stacks.txt > flamegraph.svg
```

The resulting SVG is interactive — hover over any bar to see the full call stack and time spent.

---
## Putting It All Together: A Slurm-Ready Pattern

Here is a template that generates all four exports in one profiling run.  Copy this pattern into your training scripts for batch jobs:

In [ ]:
# Complete example: all exports from one profiling run
import torch
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*Profiler clears events.*")

# Output directory (use $SLURM_SUBMIT_DIR in batch jobs)
logdir = Path("/workspace/reports/slurm_profile_example")
logdir.mkdir(parents=True, exist_ok=True)

# Reset data loader
loader_iter = iter(loader)

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    schedule=torch.profiler.schedule(wait=1, warmup=5, active=10, repeat=1),
    record_shapes=True,
    profile_memory=True,  # For memory timeline
    with_stack=True,      # For flame graphs
) as prof:
    for step in range(20):
        try:
            x, y = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            x, y = next(loader_iter)
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        
        prof.step()

# --- Generate All Exports ---
print("Saving profiling artifacts...")

# 1. Human-readable summary table
with open(logdir / "profile_summary.txt", "w") as f:
    f.write(prof.key_averages().table(sort_by="cuda_time_total", row_limit=30))
print(f"  [1/4] {logdir / 'profile_summary.txt'}")

# 2. Chrome trace for Perfetto
prof.export_chrome_trace(str(logdir / "timeline_trace.json.gz"))
print(f"  [2/4] {logdir / 'timeline_trace.json.gz'}")

# 3. Memory timeline HTML
prof.export_memory_timeline(str(logdir / "memory_timeline.html"), device="cuda:0")
print(f"  [3/4] {logdir / 'memory_timeline.html'}")

# 4. Flame graph stacks
prof.export_stacks(str(logdir / "profiler_stacks.txt"), metric="self_cuda_time_total")
print(f"  [4/4] {logdir / 'profiler_stacks.txt'}")

print(f"\nDone! Download files from {logdir} to analyze locally.")

---
## Exercise: Export Your Own Profiles

Take one of your existing training scripts and add profiling exports:

1. Add `torch.profiler.profile()` with `profile_memory=True` and `with_stack=True`
2. Generate all four export types after the profiling context closes
3. Download the Chrome trace and open it in [ui.perfetto.dev](https://ui.perfetto.dev/)
4. Compare what you see in Perfetto to what TensorBoard showed

**Bonus:** Write a small shell script that runs your training with profiling and then uses FlameGraph to automatically generate an SVG.

---
## Summary: Export Methods at a Glance

| Method | Output | View With | Use Case |
|--------|--------|-----------|----------|
| `prof.key_averages().table()` | `.txt` | Terminal / text editor | Quick summary in logs |
| `prof.export_chrome_trace()` | `.json` / `.json.gz` | [ui.perfetto.dev](https://ui.perfetto.dev/) | Detailed timeline analysis |
| `prof.export_memory_timeline()` | `.html` | Any browser | Memory debugging |
| `prof.export_stacks()` | `.txt` | FlameGraph → `.svg` | Call-stack hotspots |

All four can be generated from a single profiling run — just enable `profile_memory=True` and `with_stack=True` in your profiler configuration.

## <center><div style="text-align:center; color:#FF0000; border:3px solid red; height:80px;"><b><br/>[Next Notebook — AMP and the Limits of the Profiler](intro-amp.ipynb)</b></div></center>

---

## Links and Resources

- [Perfetto Trace Viewer](https://ui.perfetto.dev/) — recommended for viewing Chrome traces
- [FlameGraph tools](https://github.com/brendangregg/FlameGraph) — for generating flame graph SVGs
- [PyTorch Profiler documentation](https://pytorch.org/docs/stable/profiler.html)
- [PyTorch Profiler recipe](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0).